# Complete EDA Walkthrough
Exploratory Data Analysis on the Wine dataset.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_wine
from scipy import stats
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## 1. Load Dataset

In [ ]:
wine = load_wine()
df = pd.DataFrame(wine.data, columns=wine.feature_names)
df['target'] = wine.target
df['target_name'] = df['target'].map({i: name for i, name in enumerate(wine.target_names)})
print(f"Dataset: {wine.DESCR[:200]}...")
print(f"\nShape: {df.shape}")
df.head()

## 2. Data Overview

In [ ]:
print("=== Shape ===")
print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")

print("\n=== Data Types ===")
print(df.dtypes.value_counts())

print("\n=== Statistical Summary ===")
df.describe().round(2)

In [ ]:
print("=== Info ===")
df.info()
print("\n=== First Few Rows ===")
print(df.head())

## 3. Missing Value Analysis

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'count': missing, 'percentage': missing_pct})
print("Missing Values:")
print(missing_df[missing_df['count'] > 0] if missing_df['count'].sum() > 0 else "No missing values! ✓")

# Simulate and show how to analyze missing data
print("\n(Wine dataset is clean - in practice, use msno library or heatmap)")
print("Common strategies: drop, impute with mean/median/mode, or use model-based imputation")

## 4. Distribution Plots

In [ ]:
features = wine.feature_names[:6]  # First 6 features
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for ax, feat in zip(axes.flat, features):
    for target in range(3):
        subset = df[df['target'] == target][feat]
        ax.hist(subset, bins=15, alpha=0.5, label=wine.target_names[target])
    ax.set_title(feat)
    ax.legend(fontsize=8)
plt.suptitle('Feature Distributions by Class', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Remaining features
features = wine.feature_names[6:]
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for i, ax in enumerate(axes.flat):
    if i < len(features):
        for target in range(3):
            subset = df[df['target'] == target][features[i]]
            ax.hist(subset, bins=15, alpha=0.5, label=wine.target_names[target])
        ax.set_title(features[i])
    else:
        ax.set_visible(False)
axes[0,0].legend(fontsize=8)
plt.suptitle('Feature Distributions (continued)', fontsize=14)
plt.tight_layout()
plt.show()

## 5. Correlation Heatmap

In [ ]:
corr = df[wine.feature_names].corr()

plt.figure(figsize=(12, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r', 
            center=0, vmin=-1, vmax=1, square=True, linewidths=0.5)
plt.title('Feature Correlation Heatmap')
plt.tight_layout()
plt.show()

# Top correlations
upper = corr.where(mask==False)
top_corr = upper.unstack().dropna().sort_values(key=abs, ascending=False).head(10)
print("Top 10 Feature Correlations:")
for (f1, f2), val in top_corr.items():
    print(f"  {f1} ↔ {f2}: {val:.3f}")

## 6. Outlier Detection

In [ ]:
# IQR method
def detect_outliers_iqr(series):
    Q1 = series.quantile(0.25)
    Q3 = series.quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    return (series < lower) | (series > upper)

# Z-score method
def detect_outliers_zscore(series, threshold=3):
    z = np.abs(stats.zscore(series))
    return z > threshold

print("Outlier counts per feature:")
print(f"{'Feature':<25} {'IQR method':>12} {'Z-score (>3)':>14}")
print("-" * 55)
for feat in wine.feature_names:
    iqr_count = detect_outliers_iqr(df[feat]).sum()
    z_count = detect_outliers_zscore(df[feat]).sum()
    print(f"{feat:<25} {iqr_count:>12} {z_count:>14}")

In [ ]:
# Box plots for outlier visualization
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for i, ax in enumerate(axes.flat):
    if i < len(wine.feature_names[:8]):
        feat = wine.feature_names[i]
        df.boxplot(column=feat, by='target_name', ax=ax)
        ax.set_title(feat)
        ax.set_xlabel('')
plt.suptitle('Box Plots by Class (Outliers visible)', fontsize=14)
plt.tight_layout()
plt.show()

## 7. Feature-Target Relationships

In [ ]:
# ANOVA F-test for feature importance
from sklearn.feature_selection import f_classif

f_scores, p_values = f_classif(df[wine.feature_names], df['target'])

importance = pd.DataFrame({
    'feature': wine.feature_names,
    'f_score': f_scores,
    'p_value': p_values
}).sort_values('f_score', ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(importance['feature'], importance['f_score'], color='steelblue')
ax.set_xlabel('F-Score (ANOVA)')
ax.set_title('Feature Importance (F-test with Target)')
plt.tight_layout()
plt.show()
print(importance.round(4).to_string(index=False))

## 8. Pair Plots and Scatter Matrix

In [ ]:
# Select top 4 features by F-score
top_features = importance['feature'].head(4).tolist()
plot_df = df[top_features + ['target_name']]

# Pair plot
fig, axes = plt.subplots(4, 4, figsize=(14, 14))
for i, f1 in enumerate(top_features):
    for j, f2 in enumerate(top_features):
        ax = axes[i, j]
        if i == j:
            for t in range(3):
                subset = df[df['target']==t][f1]
                ax.hist(subset, bins=15, alpha=0.5)
        else:
            for t in range(3):
                subset = df[df['target']==t]
                ax.scatter(subset[f2], subset[f1], alpha=0.5, s=10)
        if j == 0: ax.set_ylabel(f1, fontsize=8)
        if i == 3: ax.set_xlabel(f2, fontsize=8)
plt.suptitle('Scatter Matrix - Top 4 Features', fontsize=14)
plt.tight_layout()
plt.show()

## 9. Statistical Tests

In [ ]:
# Normality tests (Shapiro-Wilk)
print("=== Shapiro-Wilk Normality Test ===")
print(f"{'Feature':<25} {'W-stat':>8} {'p-value':>10} {'Normal?':>8}")
print("-" * 55)
for feat in wine.feature_names[:8]:
    stat, p = stats.shapiro(df[feat])
    normal = "Yes" if p > 0.05 else "No"
    print(f"{feat:<25} {stat:>8.4f} {p:>10.4f} {normal:>8}")

print("\n=== Correlation Significance (Pearson) ===")
print(f"{'Feature Pair':<45} {'r':>6} {'p-value':>10} {'Significant?':>12}")
print("-" * 75)
for (f1, f2), corr_val in top_corr.head(5).items():
    r, p = stats.pearsonr(df[f1], df[f2])
    sig = "Yes***" if p < 0.001 else ("Yes**" if p < 0.01 else ("Yes*" if p < 0.05 else "No"))
    print(f"{f1+' ↔ '+f2:<45} {r:>6.3f} {p:>10.2e} {sig:>12}")

In [ ]:
# Kruskal-Wallis test (non-parametric ANOVA)
print("=== Kruskal-Wallis Test (Do classes differ significantly?) ===")
print(f"{'Feature':<25} {'H-stat':>8} {'p-value':>12} {'Significant?':>12}")
print("-" * 60)
for feat in wine.feature_names:
    groups = [df[df['target']==t][feat].values for t in range(3)]
    h_stat, p = stats.kruskal(*groups)
    sig = "Yes***" if p < 0.001 else "No"
    print(f"{feat:<25} {h_stat:>8.2f} {p:>12.2e} {sig:>12}")

## 10. Summary Findings and Feature Engineering Suggestions

In [ ]:
print("""
╔══════════════════════════════════════════════════════════════════╗
║                    EDA SUMMARY FINDINGS                         ║
╠══════════════════════════════════════════════════════════════════╣
║                                                                  ║
║ DATASET: Wine (178 samples, 13 features, 3 classes)             ║
║                                                                  ║
║ KEY FINDINGS:                                                    ║
║ 1. No missing values - dataset is clean                         ║
║ 2. Features have very different scales (need normalization)     ║
║ 3. Several highly correlated feature pairs exist                ║
║ 4. All features show statistically significant class separation ║
║ 5. Most features are NOT normally distributed                   ║
║ 6. Some outliers present but not extreme                        ║
║                                                                  ║
║ TOP DISCRIMINATIVE FEATURES (by F-score):                       ║
║ • flavanoids, color_intensity, proline, od280/od315             ║
║                                                                  ║
║ FEATURE ENGINEERING SUGGESTIONS:                                ║
║ 1. StandardScaler or RobustScaler (due to outliers)            ║
║ 2. Consider PCA (high correlations suggest redundancy)         ║
║ 3. Ratio features: flavanoids/phenols, color/hue              ║
║ 4. Drop low-importance features for simpler models             ║
║ 5. Polynomial features for top discriminators                   ║
║                                                                  ║
╚══════════════════════════════════════════════════════════════════╝
""")